# 部署 Real-time Endpoint

支持两种模型来源：

| 来源 | 场景 | 设置 |
|------|------|------|
| **local** | 本地跑完 `train.ipynb` Section A，有 `best.pt` | 默认，打包上传 |
| **s3** | 跑完 `train.ipynb` Section B（Training Job），模型在 S3 | 填 `MODEL_DATA_S3` |

In [ ]:
import sagemaker

sess   = sagemaker.Session()
role   = sagemaker.get_execution_role()
bucket = sess.default_bucket()

ENDPOINT_NAME = 'potato-disease-demo'

# ── 模型来源：二选一 ─────────────────────────────────────────────────
MODEL_SOURCE = 'local'   # 'local' 或 's3'

# local 模式：train.ipynb Section A 产出的 best.pt 路径
BEST_PT = 'runs/classify/train/weights/best.pt'

# s3 模式：train.ipynb Section B 训练完毕后打印的 MODEL_DATA_S3
MODEL_DATA_S3 = ''
# 例：'s3://sagemaker-us-east-1-123456789/potato-demo/training-output/potato-disease-train-xxx/output/model.tar.gz'

print(f'source={MODEL_SOURCE}, endpoint={ENDPOINT_NAME}')

In [ ]:
# local 模式：打包 best.pt + inference 脚本
if MODEL_SOURCE == 'local':
    import shutil, os, subprocess
    os.makedirs('model/code', exist_ok=True)
    shutil.copy(BEST_PT, 'model/best.pt')
    shutil.copy('../sagemaker/inference.py',   'model/code/inference.py')
    shutil.copy('../sagemaker/requirements.txt','model/code/requirements.txt')
    subprocess.run(['tar', 'czf', 'model.tar.gz', '-C', 'model', '.'], check=True)
    model_data = sess.upload_data('model.tar.gz', bucket=bucket, key_prefix='potato-demo/deploy')
    print('model_data =', model_data)
else:
    assert MODEL_DATA_S3, '请先填写 MODEL_DATA_S3'
    model_data = MODEL_DATA_S3
    print('model_data =', model_data)
    print('注意：s3 模式的 model.tar.gz 由 Training Job 打包，已内含 sagemaker/inference.py。')
    print('      若 inference.py 有改动，请用 local 模式重新打包。')

In [ ]:
# 部署 Real-time Endpoint（ml.m5.large，现场稳定无冷启动）
from sagemaker.pytorch import PyTorchModel

model = PyTorchModel(
    model_data=model_data,
    role=role,
    framework_version='2.1',
    py_version='py310',
    entry_point='inference.py',
)
predictor = model.deploy(
    endpoint_name=ENDPOINT_NAME,
    initial_instance_count=1,
    instance_type='ml.m5.large',
)
print('Endpoint 已部署：', ENDPOINT_NAME)

In [ ]:
# 演示前预热：静默调一次，让模型就绪
import json, base64, glob

sample = glob.glob('data/val/*/*')[0]
with open(sample, 'rb') as f:
    body = json.dumps({'image': base64.b64encode(f.read()).decode()})
resp = predictor.sagemaker_session.sagemaker_runtime_client.invoke_endpoint(
    EndpointName=ENDPOINT_NAME, ContentType='application/json', Body=body)
print(json.loads(resp['Body'].read()))

Endpoint 就绪后，启动前端（手动 EC2 或 `infra/` 的 CDK）。

**演示结束清理：**
```python
predictor.delete_endpoint()
```